# 07 - Push ke Hugging Face Hub

**Tujuan:** Mengunggah model standar (root repo), versi raw (`raw/`), dan laporan (`reports/`) ke satu repo.

| | |
|---|---|
| **Input** | `models/hf_export/`, `models/raw/`, `reports/` |
| **Output** | Repo model di Hugging Face |
| **Prasyarat** | Notebook 06 + akun HF + access token **write** |

> Setiap notebook berdiri sendiri: ia membaca hasil notebook sebelumnya dari Google Drive, jadi sesi Colab boleh diputus/diulang di antara notebook.

## Langkah 7.0 - Baca dulu sebelum push
Panduan lengkap: `docs/HF_NAMING_AND_PUSH_GUIDE.md`. Ringkasnya:

1. Buat token **write** di https://huggingface.co/settings/tokens
2. Nama repo: `username/nama-model`, huruf kecil, kata dipisah `-`, karakter `a-z 0-9 - _ .`, maks 96 karakter. Pola: `{backbone}-{dataset}-{tugas}`
3. Mulai dengan `private=True`; ubah ke publik setelah yakin
4. Dataset berlisensi OpenRAIL-M non-komersial; model card sudah mencantumkannya
5. **Jangan menulis token di notebook**; `notebook_login()` menanganinya

In [1]:
# --- Bootstrap (sama di setiap notebook) ---------------------------------------
# 1) Sambungkan Google Drive (tempat semua hasil disimpan permanen)
import os, sys, zipfile
from google.colab import drive
drive.mount('/content/drive')

# 2) Lokasi di Drive: folder ini sekaligus ROOT proyek (zip dan semua hasil ada di sini)
ROOT_ON_DRIVE = "/content/drive/MyDrive/models-train/mmfood-multitask"
ZIP_ON_DRIVE = f"{ROOT_ON_DRIVE}/mmfood-multitask.zip"

# 3) Folder hasil = ROOT di Drive. Harus diset SEBELUM mengimpor config
os.environ["MMFOOD_PROJECT"] = ROOT_ON_DRIVE

# 4) Ekstrak kode + dataset dari zip ke disk lokal Colab (hanya sekali per sesi)
REPO = "/content/mmfood-multitask"
if not os.path.exists(REPO):
    with zipfile.ZipFile(ZIP_ON_DRIVE) as z:
        z.extractall("/content")

# 5) Daftarkan folder src/ agar modul proyek bisa di-import
sys.path.insert(0, f"{REPO}/src")

# 6) Muat path folder (P) dan konfigurasi (CFG) dari src/config.py
from config import get_paths, load_cfg
P = get_paths()       # P.processed, P.checkpoints, P.hf_export, ... (folder dibuat otomatis)
CFG = load_cfg(P)     # nilai bawaan + perubahan dari notebook 00
print("Folder hasil:", P.project)

Mounted at /content/drive
Folder hasil: /content/drive/MyDrive/models-train/mmfood-multitask


In [2]:
# Install library yang dibutuhkan (Colab biasanya sudah punya sebagian; -q = senyap)
!pip install -q transformers accelerate huggingface_hub safetensors scikit-learn pillow tqdm matplotlib

## Langkah 7.1 - Atur nama repo dan validasi
Isi `hf_username` (dan ubah `model_name` bila perlu) lalu simpan ke konfigurasi. **Jika Anda mengubah nama setelah notebook 06**, jalankan ulang notebook 06 agar model card memuat nama yang benar.

In [3]:
import re

# Isi dengan username/organisasi Hugging Face Anda (kosongkan untuk memakai nilai dari config)
HF_USERNAME = "muhammadrafifatihulihsan"      # <- mis. "usernamekamu"
MODEL_NAME = "convnext-tiny-mmfood-multihead"       # <- mis. "convnext-tiny-mmfood-multitask"
if HF_USERNAME: CFG["hf_username"] = HF_USERNAME
if MODEL_NAME: CFG["model_name"] = MODEL_NAME

def validate_repo_name(name):
    # Kembalikan daftar masalah penamaan (kosong = valid)
    problems = []
    if not re.fullmatch(r"[A-Za-z0-9]([A-Za-z0-9._-]*[A-Za-z0-9])?", name):
        problems.append("hanya a-z A-Z 0-9 . _ - ; tidak boleh diawali/diakhiri simbol")
    if len(name) > 96: problems.append("maksimal 96 karakter")
    if "--" in name or ".." in name: problems.append("hindari '--' atau '..'")
    if name != name.lower(): problems.append("disarankan huruf kecil semua")
    return problems

assert CFG["hf_username"] != "GANTI_USERNAME_HF", "Isi hf_username dulu"
issues = validate_repo_name(CFG["model_name"])
REPO_ID = f"{CFG['hf_username']}/{CFG['model_name']}"
print("Repo ID:", REPO_ID)
print("Nama valid." if not issues else f"Masalah: {issues}")
assert not issues

# Simpan agar konsisten di notebook lain
from config import save_cfg
save_cfg(P, CFG)

Repo ID: muhammadrafifatihulihsan/convnext-tiny-mmfood-multihead
Nama valid.


## Langkah 7.2 - Login
Tempel access token (tipe write) pada kotak yang muncul.

In [4]:
from huggingface_hub import notebook_login
notebook_login()

## Langkah 7.3 - Buat repo dan upload
Tiga unggahan terpisah (tiga commit) supaya riwayat rapi:
1. `models/hf_export/` -> **root** repo (versi standar)
2. `models/raw/` -> subfolder `raw/` (arsip non-standar)
3. `reports/` -> subfolder `reports/` (metrik dan grafik)

In [5]:
from huggingface_hub import HfApi, create_repo

api = HfApi()
# Buat repo (aman dijalankan ulang: exist_ok=True)
create_repo(REPO_ID, repo_type="model", private=CFG["hf_private"], exist_ok=True)

# 1) versi STANDAR di root repo
api.upload_folder(folder_path=P.hf_export, repo_id=REPO_ID, repo_type="model",
                  commit_message="Add MM-Food multi-task model (HF standard format)")
# 2) versi RAW di subfolder raw/
api.upload_folder(folder_path=P.raw_model, path_in_repo="raw", repo_id=REPO_ID, repo_type="model",
                  commit_message="Add raw checkpoint, labels, and training config")
# 3) laporan evaluasi di subfolder reports/
api.upload_folder(folder_path=f"{P.project}/reports", path_in_repo="reports", repo_id=REPO_ID, repo_type="model",
                  commit_message="Add evaluation reports")
print("Selesai: https://huggingface.co/" + REPO_ID)

Selesai: https://huggingface.co/muhammadrafifatihulihsan/convnext-tiny-mmfood-multihead


## Langkah 7.4 - Tes muat dari Hub
Pastikan repo benar-benar bisa dipakai orang lain. Jika repo privat, pastikan sudah login.

In [6]:
from transformers import AutoModel, AutoImageProcessor

# Muat dari Hub persis seperti pengguna lain (trust_remote_code karena arsitektur kustom)
m = AutoModel.from_pretrained(REPO_ID, trust_remote_code=True)
pr = AutoImageProcessor.from_pretrained(REPO_ID)
print("Berhasil dimuat dari Hub:", type(m).__name__)

config.json:   0%|          | 0.00/76.2k [00:00<?, ?B/s]

modeling_mmfood.py:   0%|          | 0.00/6.73k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/muhammadrafifatihulihsan/convnext-tiny-mmfood-multihead:
- modeling_mmfood.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors: reconstructing file:   0%|          |  0.00B /  112MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/352 [00:00<?, ?B/s]

Berhasil dimuat dari Hub: MMFoodMultiTask


Opsional - kunci versi dengan tag agar bisa direproduksi:
```python
api.create_tag(REPO_ID, tag="v1.0", repo_type="model")
```

In [7]:
api.create_tag(REPO_ID, tag="v1.0", repo_type="model")